### Import

In [104]:
import os; import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np; import matplotlib.pyplot as plt
import gurobipy as gp; from gurobipy import GRB
from itertools import product; from tqdm import tqdm
import importlib
import functions_utils; import functions_data
import functions_optimize; import functions_eval
importlib.reload(functions_data); importlib.reload(functions_optimize)
importlib.reload(functions_eval); importlib.reload(functions_utils)
from functions_utils import *; from functions_data import *
from functions_optimize import *; from functions_eval import *
import time

S = 100
LEVEL = "high"
SEED = 10

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, _, _ = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)

CRATE = DRATE = 25

MYP = np.zeros((I, T, S)) ; MYM = np.zeros((I, T, S)) ; MZC = np.zeros((I, T, S)) ; MZD = np.zeros((I, T, S)) ; MDP = np.zeros((I, T, S)) ; MDM = np.zeros((I, T, S))  
for i, t, s in product(range(I), range(T), range(S)):
    MYP[i, t, s] = MYM[i, t, s] = MDP[i, t, s] = MDM[i, t, s] = R[i, t, s] + 0.9 * min(DRATE, K[i])
    MZC[i, t, s] = min(0.9 * CRATE, R[i, t, s])
    MZD[i, t, s] = 0.9 * DRATE

✅ 총 10개 파일을 불러왔습니다: 1201.csv, 137.csv, 281.csv, 397.csv, 401.csv, 430.csv, 514.csv, 524.csv, 775.csv, 89.csv
📊 데이터 Shape: I=10, T=24, S=100
✅ 시뮬레이션 초기화 완료: S=100, Randomness='high', Random Seed=10, M1=754.14, M2=2749.01


### Linear Decision Rule (Individual Optimization)

In [105]:
model = gp.Model("individual_LDR")
model.setParam("MIPGap", 1e-5)
model.setParam(GRB.Param.PoolSearchMode, 2)
model.setParam(GRB.Param.PoolSolutions, 2)

x_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp_ind = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_ind = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
z_ind = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc_ind = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd_ind = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

# LDR 계수 변수들
zc0_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc0") ; zc1_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc1") ; zc2_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc2")
zd0_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd0") ; zd1_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd1") ; zd2_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd2")

# Big-M 관련 변수들    
phi1_ind = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi1") ; phi2_ind = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi2") ; phi3_ind = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi3")

model.update()

obj = (gp.quicksum(P_DA[t] * x_ind[i, t] for i in range(I) for t in range(T)) + 
       gp.quicksum((1/S) * (P_RT[t, s] * yp_ind[i, t, s] - P_PN[t, s] * ym_ind[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
model.setObjective(obj, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    model.addConstr(R[i, t, s] - x_ind[i, t] == yp_ind[i, t, s] - ym_ind[i, t, s] + zc_ind[i, t, s] - zd_ind[i, t, s])
    model.addConstr(R[i, t, s] + zd_ind[i, t, s] >= yp_ind[i, t, s] + zc_ind[i, t, s])
    model.addConstr(zd_ind[i, t, s]/0.9 <= z_ind[i, t, s])
    model.addConstr(zd_ind[i, t, s]/0.9 <= DRATE)
    model.addConstr(zc_ind[i, t, s]*0.9 <= K[i] - z_ind[i, t, s])
    model.addConstr(zc_ind[i, t, s]*0.9 <= CRATE)
    model.addConstr(z_ind[i, t, s] <= K[i])
    model.addConstr(z_ind[i, t + 1, s] == z_ind[i, t, s] + 0.9 * zc_ind[i, t, s] - zd_ind[i, t, s] / 0.9)

for i, s in product(range(I), range(S)): model.addConstr(z_ind[i, 0, s] == K0[i])

for i, t, s in product(range(I), range(T), range(S)):
    model.addConstr(zc_ind[i, t, s] == zc0_ind[i, t] + zc1_ind[i, t] * R[i, t, s] + zc2_ind[i, t] * (P_RT[t, s] - P_DA[t]))
    model.addConstr(zd_ind[i, t, s] == zd0_ind[i, t] + zd1_ind[i, t] * R[i, t, s] + zd2_ind[i, t] * (P_RT[t, s] - P_DA[t]))

for i, t, s in product(range(I), range(T), range(S)):        
    model.addConstr(ym_ind[i, t, s] <= MYM[i, t, s] * phi1_ind[i, t, s]) ; model.addConstr(zc_ind[i, t, s] <= MZC[i, t, s] * (1 - phi1_ind[i, t, s]))
    model.addConstr(zc_ind[i, t, s] <= MZC[i, t, s] * phi2_ind[i, t, s]) ; model.addConstr(zd_ind[i, t, s] <= MZD[i, t, s] * (1 - phi2_ind[i, t, s]))

model.optimize()

if model.status == GRB.OPTIMAL:
    num_solutions = model.SolCount
    print(f"\n--- Solution Pool Analysis ---")
    print(f"Found {num_solutions} solutions in the pool.")

    if num_solutions > 1:
        best_obj = model.objVal
        print(f"Best objective value: {best_obj:.8f}\n")

        for i in range(num_solutions):
            model.setParam(GRB.Param.SolutionNumber, i)
            pool_obj = model.PoolObjVal
            diff = best_obj - pool_obj
            
            print(f"Solution {i}: Objective = {pool_obj},  Difference from best = {diff}")

    model.setParam(GRB.Param.SolutionNumber, 0)
    
    x_ind = np.array([[x_ind[i, t].X for t in range(T)] for i in range(I)])
    yp_ind = np.array([[[yp_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_ind = np.array([[[ym_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_ind = np.array([[[zc_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_ind = np.array([[[zd_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_ind = np.array([[[z_ind[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; original_objval = model.objVal
    phi1_ind = np.array([[[phi1_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi2_ind = np.array([[[phi2_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    
    zc0_ind = np.array([[zc0_ind[i, t].X for t in range(T)] for i in range(I)]) ; zc1_ind = np.array([[zc1_ind[i, t].X for t in range(T)] for i in range(I)]) ; zc2_ind = np.array([[zc2_ind[i, t].X for t in range(T)] for i in range(I)])
    zd0_ind = np.array([[zd0_ind[i, t].X for t in range(T)] for i in range(I)]) ; zd1_ind = np.array([[zd1_ind[i, t].X for t in range(T)] for i in range(I)]) ; zd2_ind = np.array([[zd2_ind[i, t].X for t in range(T)] for i in range(I)])
    OBJ_IND = model.objVal

Set parameter MIPGap to value 1e-05
Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 2
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 25.0.0 25A354)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-05
PoolSolutions  2
PoolSearchMode  2

Optimize a model with 337000 rows, 194680 columns and 804600 nonzeros
Model fingerprint: 0x140b3a4e
Variable types: 122680 continuous, 72000 integer (72000 binary)
Coefficient statistics:
  Matrix range     [9e-03, 8e+02]
  Objective range  [4e-01, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e-02, 8e+02]
Presolve removed 167486 rows and 44386 columns
Presolve time: 1.45s
Presolved: 169514 rows, 150294 columns, 489644 nonzeros
Variable types: 78294 continuous, 72000 integer (72000 binary)
Root relaxation presolved: 169514 rows, 150294 columns, 489644 nonzeros

Deterministic concurrent LP optimizer: prima

In [106]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 70)
print("\n[Individual]") ; print(header)
for t in range(7, 22):
    # R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_ind[:, t].sum()
    # yp_avg = np.mean([yp_ind[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_ind[:, t, s].sum() for s in range(S)])
    # zc_avg = np.mean([zc_ind[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_ind[:, t, s].sum() for s in range(S)]) 
    # z_avg = np.mean([z_ind[:, t, s].sum() for s in range(S)])

    i=0
    R_avg = np.mean([R[i, t, s] for s in range(S)]) ; x_sum = x_ind[i, t]
    yp_avg = np.mean([yp_ind[i, t, s] for s in range(S)]) ; ym_avg = np.mean([ym_ind[i, t, s] for s in range(S)])
    zc_avg = np.mean([zc_ind[i, t, s] for s in range(S)]) ; zd_avg = np.mean([zd_ind[i, t, s] for s in range(S)]) 
    z_avg = np.mean([z_ind[i, t, s] for s in range(S)])

    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[Individual]
 t |        R        x       y+       y-       zc       zd        z
----------------------------------------------------------------------
 7 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00
 8 |    18.36     0.00    14.74     0.00     3.62     0.00     0.00
 9 |     3.16     0.00     2.41     0.00     0.75     0.00     3.26
10 |    36.83     0.00    29.27     0.00     7.56     0.00     3.93
11 |    62.98     0.00    49.92     0.00    13.07     0.00    10.73
12 |   145.05     0.00   122.55     0.00    22.50     0.00    22.49
13 |   196.67     0.00   212.63     0.00     0.00    15.97    42.74
14 |   208.13   105.68   106.51     4.06     0.00     0.00    25.00
15 |   171.49     0.00   193.99     0.00     0.00    22.50    25.00
16 |    20.38     8.10    12.54     0.26     0.00     0.00     0.00
17 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00
18 |    79.60     0.00    79.60     0.00     0.00     0.00     0.00
19 |    53.99    25.02    30.27

In [107]:
print("\n=== 모든 LDR 계수 (i, t별) ===")
print("Individual | Time | Variable | 상수항    | R계수    | (RT_DA)계수    ")
print("-" * 50)

for t in range(9,20):
    for i in range(I):
        print(f"{i:10d} | {t:4d} | zc       | {zc0_ind[i,t]:8.4f} | {zc1_ind[i,t]:7.4f} | {zc2_ind[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | zd       | {zd0_ind[i,t]:8.4f} | {zd1_ind[i,t]:7.4f} | {zd2_ind[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | x (1st)  | {x_ind[i,t]:8.4f} |    -    ")
        print("-" * 50)


=== 모든 LDR 계수 (i, t별) ===
Individual | Time | Variable | 상수항    | R계수    | (RT_DA)계수    
--------------------------------------------------
         0 |    9 | zc       |   0.7456 |  0.0000 |  0.0000
         0 |    9 | zd       |   0.0000 |  0.0000 |  0.0000
         0 |    9 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         1 |    9 | zc       |  11.3378 |  0.0000 |  0.0000
         1 |    9 | zd       |   0.0000 |  0.0000 |  0.0000
         1 |    9 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         2 |    9 | zc       |   7.1541 |  0.0000 |  0.0000
         2 |    9 | zd       |   0.0000 |  0.0000 |  0.0000
         2 |    9 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         3 |    9 | zc       |   3.8290 |  0.0947 |  0.0002
         3 |    9 | zd       |   0.0000 |  0.0000 |  0.0000
         3 |    9 | x (1st)  |   0.0000 |    -    
----------------------

### Linear Decision Rule (Holistic Optimization) + Convex Hull Pricing

In [108]:
model = gp.Model("holistic_LDR")
model.setParam("MIPGap", 1e-5)
model.setParam(GRB.Param.PoolSearchMode, 2)
model.setParam(GRB.Param.PoolSolutions, 1)

x_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x") ; yp_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z_hol = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z") ; zc_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

L1_YPYM_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L1_YPYM_HOL") ; L2_YPYM_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L2_YPYM_HOL") ; L3_YPYM_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L3_YPYM_HOL") 
L1_DPDM_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L1_DPDM_HOL") ; L2_DPDM_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L2_DPDM_HOL") ; L3_DPDM_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L3_DPDM_HOL")
L1_YPDM_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L1_YPDM_HOL") ; L2_YPDM_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L2_YPDM_HOL") ; L3_YPDM_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L3_YPDM_HOL") 
L1_YMDP_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L1_YMDP_HOL") ; L2_YMDP_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L2_YMDP_HOL") ; L3_YMDP_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L3_YMDP_HOL") 
L1_YMZC_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L1_YMZC_HOL") ; L2_YMZC_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L2_YMZC_HOL") ; L3_YMZC_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L3_YMZC_HOL") 
L1_DMZC_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L1_DMZC_HOL") ; L2_DMZC_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L2_DMZC_HOL") ; L3_DMZC_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L3_DMZC_HOL")
L1_ZCZD_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L1_ZCZD_HOL") ; L2_ZCZD_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L2_ZCZD_HOL") ; L3_ZCZD_HOL = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="L3_ZCZD_HOL") 

zc0_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc0") ; zc1_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc1") ; zc2_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc2")
zd0_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd0") ; zd1_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd1") ; zd2_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd2")

model.update()

obj = (gp.quicksum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) + 
       gp.quicksum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
model.setObjective(obj, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    model.addConstr(R[i, t, s] - x_hol[i, t] == yp_hol[i, t, s] - ym_hol[i, t, s] + dp_hol[i, t, s] - dm_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
    model.addConstr(R[i, t, s] + zd_hol[i, t, s] >= yp_hol[i, t, s] + dp_hol[i, t, s] + zc_hol[i, t, s])
    model.addConstr(zd_hol[i, t, s]/0.9 <= z_hol[i, t, s])
    model.addConstr(zd_hol[i, t, s]/0.9 <= DRATE)
    model.addConstr(zc_hol[i, t, s]*0.9 <= K[i] - z_hol[i, t, s])
    model.addConstr(zc_hol[i, t, s]*0.9 <= CRATE)
    model.addConstr(z_hol[i, t, s] <= K[i])
    model.addConstr(z_hol[i, t + 1, s] == z_hol[i, t, s] + 0.9 * zc_hol[i, t, s] - zd_hol[i, t, s] / 0.9)
for i, s in product(range(I), range(S)): model.addConstr(z_hol[i, 0, s] == K0[i])

balance_constraints = {}
for t, s in product(range(T), range(S)):
    balance_constraints[t, s] = model.addConstr(gp.quicksum(dp_hol[i, t, s] for i in range(I)) == gp.quicksum(dm_hol[i, t, s] for i in range(I)), name=f"balance_{t}_{s}")

for i, t, s in product(range(I), range(T), range(S)):
    model.addConstr(zc_hol[i, t, s] == zc0_hol[i, t] + zc1_hol[i, t] * R[i, t, s] + zc2_hol[i, t] * (P_RT[t, s] - P_DA[t]))
    model.addConstr(zd_hol[i, t, s] == zd0_hol[i, t] + zd1_hol[i, t] * R[i, t, s] + zd2_hol[i, t] * (P_RT[t, s] - P_DA[t]))

for i, t, s in product(range(I), range(T), range(S)):
    # model.addConstr(L1_YPYM_HOL[i, t, s] + L2_YPYM_HOL[i, t, s] + L3_YPYM_HOL[i, t, s] == 1) ; model.addConstr(yp_hol[i, t, s] == L2_YPYM_HOL[i, t, s] * MYP[i, t, s]) ; model.addConstr(ym_hol[i, t, s] == L3_YPYM_HOL[i, t, s] * MYM[i, t, s])
    model.addConstr(L1_DPDM_HOL[i, t, s] + L2_DPDM_HOL[i, t, s] + L3_DPDM_HOL[i, t, s] == 1) ; model.addConstr(dp_hol[i, t, s] == L2_DPDM_HOL[i, t, s] * MDP[i, t, s]) ; model.addConstr(dm_hol[i, t, s] == L3_DPDM_HOL[i, t, s] * MDM[i, t, s])
    model.addConstr(L1_YPDM_HOL[i, t, s] + L2_YPDM_HOL[i, t, s] + L3_YPDM_HOL[i, t, s] == 1) ; model.addConstr(yp_hol[i, t, s] == L2_YPDM_HOL[i, t, s] * MYP[i, t, s]) ; model.addConstr(dm_hol[i, t, s] == L3_YPDM_HOL[i, t, s] * MDM[i, t, s])
    model.addConstr(L1_YMDP_HOL[i, t, s] + L2_YMDP_HOL[i, t, s] + L3_YMDP_HOL[i, t, s] == 1) ; model.addConstr(ym_hol[i, t, s] == L2_YMDP_HOL[i, t, s] * MYM[i, t, s]) ; model.addConstr(dp_hol[i, t, s] == L3_YMDP_HOL[i, t, s] * MDP[i, t, s])
    model.addConstr(L1_YMZC_HOL[i, t, s] + L2_YMZC_HOL[i, t, s] + L3_YMZC_HOL[i, t, s] == 1) ; model.addConstr(ym_hol[i, t, s] == L2_YMZC_HOL[i, t, s] * MYM[i, t, s]) ; model.addConstr(zc_hol[i, t, s] == L3_YMZC_HOL[i, t, s] * MZC[i, t, s])
    model.addConstr(L1_DMZC_HOL[i, t, s] + L2_DMZC_HOL[i, t, s] + L3_DMZC_HOL[i, t, s] == 1) ; model.addConstr(dm_hol[i, t, s] == L2_DMZC_HOL[i, t, s] * MDM[i, t, s]) ; model.addConstr(zc_hol[i, t, s] == L3_DMZC_HOL[i, t, s] * MZC[i, t, s])
    model.addConstr(L1_ZCZD_HOL[i, t, s] + L2_ZCZD_HOL[i, t, s] + L3_ZCZD_HOL[i, t, s] == 1) ; model.addConstr(zc_hol[i, t, s] == L2_ZCZD_HOL[i, t, s] * MZC[i, t, s]) ; model.addConstr(zd_hol[i, t, s] == L3_ZCZD_HOL[i, t, s] * MZD[i, t, s])

model.optimize()

Set parameter MIPGap to value 1e-05
Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 25.0.0 25A354)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-05
PoolSolutions  1
PoolSearchMode  2

Optimize a model with 675400 rows, 674680 columns and 1731500 nonzeros
Model fingerprint: 0x2086b181
Coefficient statistics:
  Matrix range     [9e-03, 8e+02]
  Objective range  [4e-01, 2e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e-02, 8e+02]
Presolve time: 0.66s
Presolved: 214444 rows, 326913 columns, 898697 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 3.11s
Elapsed ordering time = 5s
Ordering time: 6.57s

Barrier statistics:
 AA' NZ     : 4.775e+06
 Factor NZ  : 5.422e+07 (roughly 700 MB of memory)
 Factor Ops : 5.481e+10 (les

In [109]:
if model.status == GRB.OPTIMAL:
    num_solutions = model.SolCount
    print(f"\n--- Solution Pool Analysis ---")
    print(f"Found {num_solutions} solutions in the pool.")

    if num_solutions > 1:
        best_obj = model.objVal
        print(f"Best objective value: {best_obj:.8f}\n")

        for i in range(num_solutions):
            model.setParam(GRB.Param.SolutionNumber, i)
            pool_obj = model.PoolObjVal
            diff = best_obj - pool_obj
            print(f"Solution {i}: Objective = {pool_obj},  Difference from best = {diff}")

    model.setParam(GRB.Param.SolutionNumber, 0)
    
    x_hol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
    yp_hol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_hol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_hol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_hol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_hol = np.array([[[zc_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_hol = np.array([[[zd_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_hol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; original_objval = model.objVal
    OBJ_HOL = model.objVal
    
    zc0_hol = np.array([[zc0_hol[i, t].X for t in range(T)] for i in range(I)]) ; zc1_hol = np.array([[zc1_hol[i, t].X for t in range(T)] for i in range(I)]) ; zc2_hol = np.array([[zc2_hol[i, t].X for t in range(T)] for i in range(I)])
    zd0_hol = np.array([[zd0_hol[i, t].X for t in range(T)] for i in range(I)]) ; zd1_hol = np.array([[zd1_hol[i, t].X for t in range(T)] for i in range(I)]) ; zd2_hol = np.array([[zd2_hol[i, t].X for t in range(T)] for i in range(I)])


--- Solution Pool Analysis ---
Found 1 solutions in the pool.


In [110]:
if model.status == GRB.OPTIMAL:        
    try:
        lambda_dual = {}
        for t, s in product(range(T), range(S)): 
            lambda_dual[t, s] = balance_constraints[t, s].Pi
        print("Direct dual extraction successful!")
        
    except AttributeError:
        print("\nDirect dual extraction failed. Using Model.fixed() method...")
        
        fixed_model = model.fixed()
        
        fixed_x_hol = {(i, t): fixed_model.getVarByName(f"x[{i},{t}]") for i, t in product(range(I), range(T))}
        fixed_yp_hol = {(i, t, s): fixed_model.getVarByName(f"yp[{i},{t},{s}]") for i, t, s in product(range(I), range(T), range(S))}
        fixed_ym_hol = {(i, t, s): fixed_model.getVarByName(f"ym[{i},{t},{s}]") for i, t, s in product(range(I), range(T), range(S))}

        linear_obj_for_fixed_model = (
            gp.quicksum(P_DA[t] * fixed_x_hol[i, t] for i, t in product(range(I), range(T))) +
            gp.quicksum((1/S) * (P_RT[t, s] * fixed_yp_hol[i, t, s] - P_PN[t, s] * fixed_ym_hol[i, t, s]) 
                         for i, t, s in product(range(I), range(T), range(S)))
        )
        
        reg_fixed = gp.quicksum(fixed_x_hol[i, t] * fixed_x_hol[i, t] for i, t in product(range(I), range(T)))
        regularized_obj_fixed = linear_obj_for_fixed_model - epsilon * reg_fixed
        # regularized_obj_fixed = linear_obj_for_fixed_model
        
        fixed_model.setParam("Method", 1)
        fixed_model.setParam("MIPGap", 1e-5)
        fixed_model.setParam(GRB.Param.PoolSearchMode, 2)
        fixed_model.setParam(GRB.Param.PoolSolutions, 2)
        fixed_model.setObjective(regularized_obj_fixed, GRB.MAXIMIZE)
        fixed_model.optimize()
        
        if fixed_model.status == GRB.OPTIMAL:
            original_obj_val = (
                sum(P_DA[t] * x_hol[i, t] for i, t in product(range(I), range(T))) +
                sum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) 
                    for i, t, s in product(range(I), range(T), range(S)))
                - epsilon * sum(x_hol[i, t] * x_hol[i, t] for i, t in product(range(I), range(T)))
            )
            
            fixed_obj = fixed_model.objVal
            obj_diff = abs(original_obj_val - fixed_obj)

            print(f"Original Objective: {original_obj_val:.6f}")
            print(f"Fixed Model Objective: {fixed_obj:.6f}")
            print(f"Difference: {obj_diff:.10f}")
            
            if obj_diff < 1e-6:
                print("✅ Objective values match! Fixed model is consistent.")
            else:
                print("⚠️ Warning: Objective values don't match.")

            num_solutions = fixed_model.SolCount
            print(f"\n--- Fixed Model Solution Pool Analysis ---")
            print(f"Found {num_solutions} solutions in the pool.")

            if num_solutions > 1:
                best_obj = fixed_model.objVal
                print(f"Best objective value: {best_obj:.8f}\n")
                for i in range(num_solutions):
                    fixed_model.setParam(GRB.Param.SolutionNumber, i)
                    pool_obj = fixed_model.PoolObjVal
                    diff = abs(best_obj - pool_obj)
                    print(f"Solution {i}: Objective = {pool_obj:.8f},  Difference = {diff:.8f}")
                fixed_model.setParam(GRB.Param.SolutionNumber, 0)

            print("\n=== Solution Comparison ===")
            fixed_vars = {var.VarName: var.X for var in fixed_model.getVars()}
            
            print("=== x values by individual ===")
            print("Individual | Time | Original x | Fixed x | Difference")
            print("-" * 55)
            
            max_x_diff = 0
            for i in range(I):
                for t in range(T):
                    original_x = x_hol[i, t]
                    fixed_x = fixed_vars.get(f"x[{i},{t}]", 0)
                    diff = abs(original_x - fixed_x)
                    max_x_diff = max(max_x_diff, diff)
                    if original_x != 0:
                        print(f"{i:10d} | {t:4d} | {original_x:10.6f} | {fixed_x:7.6f} | {diff:10.8f}")
            
            print("\n=== yp values sum over i (scenario average) ===")
            print("Time | Original yp_sum | Fixed yp_sum | Difference")
            print("-" * 52)
            max_yp_diff = 0
            for t in range(14,16):
                original_yp_sum = sum(sum(yp_hol[i, t, s] for i in range(I)) for s in range(S)) / S
                fixed_yp_sum = sum(sum(fixed_vars.get(f"yp[{i},{t},{s}]", 0) for i in range(I)) for s in range(S)) / S
                diff = abs(original_yp_sum - fixed_yp_sum)
                max_yp_diff = max(max_yp_diff, diff)
                print(f"{t:4d} | {original_yp_sum:14.6f} | {fixed_yp_sum:12.6f} | {diff:10.8f}")

        lambda_dual = np.zeros((T, S))
        if fixed_model.status == GRB.OPTIMAL:
            for t, s in product(range(T), range(S)):
                constr_name = f"balance_{t}_{s}"
                try:
                    constr = fixed_model.getConstrByName(constr_name)
                    if constr is not None:
                        lambda_dual[t, s] = constr.Pi
                    else:
                        lambda_dual[t, s] = np.nan
                except:
                    lambda_dual[t, s] = np.nan 
                    
            print("\nModel.fixed() dual extraction successful!")
        else:
            print("Fixed model optimization failed. Setting dual to zeros.")
            lambda_dual = np.zeros((T, S))

        print("\nInternal Settlement Prices (Dual Variables):")
        for t, s in product(range(T), range(1,2)):
            print(f"λ_{t}(ξ_{s}) = {lambda_dual[t, s]:.4f}")
            
else:
    print("No optimal solution found.")
    lambda_dual = {(t, s): np.nan for t in range(T) for s in range(S)}
    x_hol = yp_hol = ym_hol = dp_hol = dm_hol = z_hol = zc_hol = zd_hol = None
    original_objval = None

Direct dual extraction successful!


In [111]:
print("\n=== 모든 LDR 계수 (i, t별) ===")
print("Individual | Time | Variable | 상수항     | R계수    | (RT_DA)계수    ")
print("-" * 50)

for t in range(7,20):
    for i in range(I):
        print(f"{i:10d} | {t:4d} | zc       | {zc0_hol[i,t]:8.4f} | {zc1_hol[i,t]:7.4f} | {zc2_hol[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | zd       | {zd0_hol[i,t]:8.4f} | {zd1_hol[i,t]:7.4f} | {zd2_hol[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | x (1st)  | {x_hol[i,t]:8.4f} |    -    ")
        print("-" * 50)


=== 모든 LDR 계수 (i, t별) ===
Individual | Time | Variable | 상수항     | R계수    | (RT_DA)계수    
--------------------------------------------------
         0 |    7 | zc       |   0.0000 |  0.0000 |  0.0000
         0 |    7 | zd       |   0.0000 |  0.0000 |  0.0000
         0 |    7 | x (1st)  |  18.7455 |    -    
--------------------------------------------------
         1 |    7 | zc       |   0.9708 |  0.0000 |  0.0000
         1 |    7 | zd       |   0.0000 |  0.0000 |  0.0000
         1 |    7 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         2 |    7 | zc       |   2.1509 |  0.0000 |  0.0000
         2 |    7 | zd       |   0.0000 |  0.0000 |  0.0000
         2 |    7 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         3 |    7 | zc       |   0.0000 |  1.0000 |  0.0000
         3 |    7 | zd       |   0.0000 |  0.0000 |  0.0000
         3 |    7 | x (1st)  |   0.0000 |    -    
---------------------

In [112]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 90)
print("\n[HOLISTIC]") ; print(header)
for t in range(7, 22):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_hol[:, t].sum()
    yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[HOLISTIC]
 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 7 |    45.76    18.75    13.21     0.35    18.39    18.39    14.16     0.00     8.29
 8 |   100.91    44.43    30.00     0.60    43.20    43.20    27.08     0.00    21.03
 9 |   283.74   178.65    60.43     5.34   155.39   155.39    50.00     0.00    45.40
10 |   700.61   438.59   197.83     6.56   286.12   286.12    70.76     0.00    90.40
11 |   900.33    45.21   690.20     0.04     2.54     2.54   164.95     0.00   154.08
12 |  1247.34     0.00  1129.31     0.00     0.00     0.00   118.32     0.29   302.54
13 |  1732.65     0.00  1880.21     0.00     0.00     0.00     0.00   147.56   408.70
14 |  1820.46  1420.20   415.06    34.34   934.81   934.81    20.17     0.63   244.75
15 |  1152.93     0.00  1376.99     0.00     0.00     0.00     0.00   224.07   262.20
16 |   969.38   748.42   237.85    16

### Individual Replay

In [113]:
lambda_rep = np.zeros((T, S))
data = []
# eps = 0.0000001
eps = 0

for t in range(T):
    for s in range(S):
        lambda_rep[t, s] = lambda_dual[t, s]
        
        if abs(lambda_rep[t, s] - (-P_RT[t, s] / S)) < 0.01:
            lambda_rep[t, s] -= eps / S
            
        elif abs(lambda_rep[t, s] - (-P_PN[t, s] / S)) < 0.01:
            lambda_rep[t, s] += eps / S
    
    s_fixed = 29
    data.append({
        'Time': t, 
        'P_DA': round(P_DA[t], 2), 
        'P_RT_avg': round(P_RT[t, s_fixed], 5), 
        'Lambda': round(-lambda_rep[t, s_fixed] * S, 5), 
        'P_PN_avg': round(P_PN[t, s_fixed], 4)
    })

pd.DataFrame(data)

,Time,P_DA,P_RT_avg,Lambda,P_PN_avg
0,0,106.720,69.042,213.444,213.444
1,1,93.130,61.829,61.829,186.256
2,2,86.030,86.396,86.396,172.791
3,3,82.960,66.274,165.928,165.928
4,4,82.730,80.374,80.374,165.452
5,5,85.950,56.294,171.892,171.892
6,6,93.550,79.501,128.401,187.096
7,7,102.330,102.721,102.721,205.442
8,8,122.120,86.302,86.302,244.244
9,9,132.150,74.611,74.611,264.292


In [114]:
import pandas as pd
import numpy as np


T = 24
num_scenarios = 100
S_scalar = 100 # lambda_dual에 곱해질 상수 S

data_for_csv = []

# 모든 시간(t)과 시나리오(s)에 대해 반복합니다.
for t in range(T):
    for s in range(num_scenarios):
        row = {
            't': t,
            's': s,
            'P_DA': P_DA[t], # P_DA[t]는 해당 t의 모든 s에 대해 동일한 값이 저장됩니다.
            'P_RT': P_RT[t, s],
            'P_PN': P_PN[t, s],
            'P_IN': -lambda_dual[t, s] * S_scalar
        }
        data_for_csv.append(row)

df = pd.DataFrame(data_for_csv)

output_filename = f'optimization_results_{SEED}.csv'
df.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"✅ 데이터가 '{output_filename}' 파일로 성공적으로 저장되었습니다.")

✅ 데이터가 'optimization_results_10.csv' 파일로 성공적으로 저장되었습니다.


In [115]:
model = gp.Model("DER_Individual_Replay")
model.setParam("MIPGap", 1e-5)
model.setParam(GRB.Param.PoolSearchMode, 2)
model.setParam(GRB.Param.PoolSolutions, 2)

x = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym") 
dp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm") 
z = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
phi1 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi1") ; phi2 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi2") 
phi3 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi3") ; phi4 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
phi5 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi5") ; phi6 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi6") ; phi7 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi7")
zc0 = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc0") ; zc1 = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zcR") ; zc2 = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc2")
zd0 = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd0") ; zd1 = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zdR") ; zd2 = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd2")

model.update()

obj = (
    gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) + 
    gp.quicksum((1/S) * (
        P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s]
    ) for i in range(I) for t in range(T) for s in range(S)) +
    gp.quicksum(lambda_rep[t, s] * (
        gp.quicksum(dm[i, t, s] for i in range(I)) - gp.quicksum(dp[i, t, s] for i in range(I))
    ) for t in range(T) for s in range(S))
)
model.setObjective(obj, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    model.addConstr(zc[i, t, s] == zc0[i, t] + zc1[i, t] * R[i, t, s] + zc2[i, t] * (P_RT[t, s] - P_DA[t]))
    model.addConstr(zd[i, t, s] == zd0[i, t] + zd1[i, t] * R[i, t, s] + zd2[i, t] * (P_RT[t, s] - P_DA[t]))

for i, t, s in product(range(I), range(T), range(S)):
    model.addConstr(R[i, t, s] - x[i, t] == yp[i, t, s] - ym[i, t, s] + dp[i, t, s] - dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
    model.addConstr(R[i, t, s] + zd[i, t, s] >= yp[i, t, s] + dp[i, t, s] + zc[i, t, s])
    model.addConstr(zd[i, t, s]/0.9 <= z[i, t, s]) ; model.addConstr(zc[i, t, s]*0.9 <= K[i] - z[i, t, s]) ; model.addConstr(z[i, t, s] <= K[i])
    model.addConstr(zd[i, t, s]/0.9 <= DRATE) ; model.addConstr(zc[i, t, s]*0.9 <= CRATE)
    model.addConstr(z[i, t + 1, s] == z[i, t, s] + 0.9 * zc[i, t, s] - zd[i, t, s] / 0.9)
for i, s in product(range(I), range(S)): model.addConstr(z[i, 0, s] == K0[i])

for i, t, s in product(range(I), range(T), range(S)):       
    model.addConstr(dp[i, t, s] <= MDP[i, t, s] * phi2[i, t, s]) ; model.addConstr(dm[i, t, s] <= MDM[i, t, s] * (1 - phi2[i, t, s]))
    model.addConstr(yp[i, t, s] <= MYP[i, t, s] * phi3[i, t, s]) ; model.addConstr(dm[i, t, s] <= MDM[i, t, s] * (1 - phi3[i, t, s]))
    model.addConstr(ym[i, t, s] <= MYM[i, t, s] * phi4[i, t, s]) ; model.addConstr(dp[i, t, s] <= MDP[i, t, s] * (1 - phi4[i, t, s]))
    model.addConstr(ym[i, t, s] <= MYM[i, t, s] * phi5[i, t, s]) ; model.addConstr(zc[i, t, s] <= MZC[i, t, s] * (1 - phi5[i, t, s]))
    model.addConstr(dm[i, t, s] <= MDM[i, t, s] * phi6[i, t, s]) ; model.addConstr(zc[i, t, s] <= MZC[i, t, s] * (1 - phi6[i, t, s]))
    model.addConstr(zc[i, t, s] <= MZC[i, t, s] * phi7[i, t, s]) ; model.addConstr(zd[i, t, s] <= MZD[i, t, s] * (1 - phi7[i, t, s]))

model.optimize()

if model.status == GRB.OPTIMAL:
    num_solutions = model.SolCount
    print(f"\n--- Solution Pool Analysis ---")
    print(f"Found {num_solutions} solutions in the pool.")

    if num_solutions > 1:
        best_obj = model.objVal
        print(f"Best objective value: {best_obj:.8f}\n")

        for i in range(num_solutions):
            model.setParam(GRB.Param.SolutionNumber, i)
            pool_obj = model.PoolObjVal
            diff = best_obj - pool_obj
            
            print(f"Solution {i}: Objective = {pool_obj},  Difference from best = {diff}")

    model.setParam(GRB.Param.SolutionNumber, 0)
    
    print(f"Optimal solution found! Objective value: {model.objVal}")
else:
    print("No optimal solution found.")

x_re = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
yp_re = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_re = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
dp_re = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_re = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
z_re = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
zc_re = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_re = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
zc0_re = np.array([[zc0[i, t].X for t in range(T)] for i in range(I)]) ; zd0_re = np.array([[zd0[i, t].X for t in range(T)] for i in range(I)])
zc1_re = np.array([[zc1[i, t].X for t in range(T)] for i in range(I)]) ; zd1_re = np.array([[zd1[i, t].X for t in range(T)] for i in range(I)])
zc2_re = np.array([[zc2[i, t].X for t in range(T)] for i in range(I)]) ; zd2_re = np.array([[zd2[i, t].X for t in range(T)] for i in range(I)])
OBJ_RE = model.objVal

Set parameter MIPGap to value 1e-05
Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 2


KeyboardInterrupt: 

In [ ]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 90)
print("\n[REPLAY]") ; print(header)
for t in range(T):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_re[:, t].sum()
    yp_avg = np.mean([yp_re[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_re[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_re[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_re[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_re[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_re[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_re[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")

print("\n[HOLISTIC]") ; print(header)
for t in range(T):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_hol[:, t].sum()
    yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[REPLAY]
 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 0 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00
 1 |     0.12     0.00     0.02     0.00     0.10     0.00     0.00     0.00     0.00
 2 |     0.26     0.00     0.03     0.00     0.09     0.00     0.15     0.00     0.00
 3 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.13
 4 |     0.79     0.00     0.00     0.00     0.00     0.00     0.79     0.00     0.13
 5 |     0.85     0.00     0.00     0.00     0.00     0.00     0.85     0.00     0.84
 6 |     7.90     0.00     1.45     0.00     0.26     0.00     6.19     0.00     1.61
 7 |    45.55    22.50    27.98     0.00     2.34    22.50    15.22     0.00     7.18
 8 |   100.06    45.51    67.20     3.37     6.91    40.88    24.68     0.00    20.88
 9 |   277.73   144.35     3.37     7.3

In [ ]:
print(round(x_ind[:,:].sum(),2), round(x_re[:,:].sum(),2), round(x_hol[:,:].sum(),2))

1602.91 3203.67 3357.49


In [ ]:
print("\n=== 모든 LDR 계수 (i, t별) ===")
print("Individual | Time | Variable | 상수항     | R계수    | (RT_DA)계수    ")
print("-" * 50)

for t in range(T):
    for i in range(I):
        print(f"{i:10d} | {t:4d} | zc       | {zc0_re[i,t]:8.4f} | {zc1_re[i,t]:7.4f} | {zc2_re[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | zd       | {zd0_re[i,t]:8.4f} | {zd1_re[i,t]:7.4f} | {zd2_re[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | x (1st)  | {x_re[i,t]:8.4f} |    -    ")
        print("-" * 50)


=== 모든 LDR 계수 (i, t별) ===
Individual | Time | Variable | 상수항     | R계수    | (RT_DA)계수    
--------------------------------------------------
         0 |    0 | zc       |   0.0000 |  0.0000 |  0.0000
         0 |    0 | zd       |   0.0000 |  0.0000 |  0.0000
         0 |    0 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         1 |    0 | zc       |   0.0000 |  0.0000 |  0.0000
         1 |    0 | zd       |   0.0000 |  0.0000 |  0.0000
         1 |    0 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         2 |    0 | zc       |   0.0000 |  0.0000 |  0.0000
         2 |    0 | zd       |   0.0000 |  0.0000 |  0.0000
         2 |    0 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         3 |    0 | zc       |   0.0000 |  0.0000 |  0.0000
         3 |    0 | zd       |   0.0000 |  0.0000 |  0.0000
         3 |    0 | x (1st)  |   0.0000 |    -    
---------------------

In [ ]:
print("="*50) ; print("AGGREGATOR LOSS ANALYSIS") ; print("="*50)
total_losses = []
for t in range(T):
    scenario_losses = []
    
    for s in range(S):
        total_supply = np.sum(dp_re[:, t, s])
        total_demand = np.sum(dm_re[:, t, s])
        lambda_price = -lambda_rep[t, s] * S
        
        loss = (total_demand - total_supply) * (lambda_price - P_RT[t, s])
        scenario_losses.append(loss)
    
    avg_loss = np.mean(scenario_losses)
    total_losses.append(avg_loss)
overall_avg_loss = np.mean(total_losses)
total_loss = np.sum(total_losses)
print("Individual Participation Profit", OBJ_IND)
print("Expected Replay Profit", OBJ_RE)
print(f"Total loss across all time periods: {total_loss:.2f}")
print("Realized Profit", OBJ_RE + total_loss)
print("Holistic Profit", OBJ_HOL)
print(); print("="*60) ; print("INDIVIDUAL PROFIT ANALYSIS") ; print("="*60)
profit_ind = np.zeros(I) ; profit_re = np.zeros(I) ; profit_hol = np.zeros(I) ; profit_re_adjusted = np.zeros(I)

# 가격차이 고려한 가중치 계산
price_weighted_usage = np.zeros(I)
for i in range(I):
    for t in range(T):
        for s in range(S):
            lambda_price = -lambda_rep[t, s] * S
            # dm * (P_PN - lambda) + dp * (lambda - P_RT)
            dm_contribution = dm_re[i, t, s] * (lambda_price)
            dp_contribution = dp_re[i, t, s] * (lambda_price)
            price_weighted_usage[i] += dm_contribution + dp_contribution

total_price_weighted_usage = np.sum(price_weighted_usage)
print(f"Total price-weighted usage: {total_price_weighted_usage:.2f}")

for i in range(I):
    profit_ind[i] = 0
    for t in range(T):
        profit_ind[i] += P_DA[t] * x_ind[i, t]
        profit_ind[i] += np.mean([P_RT[t, s] * yp_ind[i, t, s] for s in range(S)])
        profit_ind[i] -= np.mean([P_PN[t, s] * ym_ind[i, t, s] for s in range(S)])
    
    profit_re[i] = 0
    for t in range(T):
        profit_re[i] += P_DA[t] * x_re[i, t]
        profit_re[i] += np.mean([P_RT[t, s] * yp_re[i, t, s] for s in range(S)])
        profit_re[i] -= np.mean([P_PN[t, s] * ym_re[i, t, s] for s in range(S)])
        lambda_price = -lambda_rep[t, :] * S
        profit_re[i] += np.mean([lambda_price[s] * dp_re[i, t, s] for s in range(S)])
        profit_re[i] -= np.mean([lambda_price[s] * dm_re[i, t, s] for s in range(S)])
    
    profit_hol[i] = 0
    for t in range(T):
        profit_hol[i] += P_DA[t] * x_hol[i, t]
        profit_hol[i] += np.mean([P_RT[t, s] * yp_hol[i, t, s] for s in range(S)])
        profit_hol[i] -= np.mean([P_PN[t, s] * ym_hol[i, t, s] for s in range(S)])
        lambda_price = -lambda_rep[t, :] * S
        profit_hol[i] += np.mean([lambda_price[s] * dp_hol[i, t, s] for s in range(S)])
        profit_hol[i] -= np.mean([lambda_price[s] * dm_hol[i, t, s] for s in range(S)])

# 가격차이 고려한 loss 분배
loss_per_player = np.zeros(I)
for i in range(I):
    if total_price_weighted_usage > 0:
        loss_per_player[i] = total_loss * (price_weighted_usage[i] / total_price_weighted_usage)
    else:
        loss_per_player[i] = total_loss / I  # fallback to equal distribution
    profit_re_adjusted[i] = profit_re[i] + loss_per_player[i]

print(f"{'Player':<8} {'Individual':<12} {'Replay':<12} {'Re+Loss':<12} {'Holistic':<12} {'Price Weight':<12} {'Loss Share':<12} {'Re+Loss-Ind':<12}")
print("-" * 110)
total_ind = 0 ; total_re = 0 ; total_hol = 0 ; total_re_adj = 0
for i in range(I):
    diff_adj_ind = profit_re_adjusted[i] - profit_ind[i]
    
    print(f"{i:<8} {profit_ind[i]:<12.2f} {profit_re[i]:<12.2f} {profit_re_adjusted[i]:<12.2f} {profit_hol[i]:<12.2f} {price_weighted_usage[i]:<12.2f} {loss_per_player[i]:<12.2f} {diff_adj_ind:<12.2f}")
    
    total_ind += profit_ind[i]
    total_re += profit_re[i]
    total_hol += profit_hol[i]
    total_re_adj += profit_re_adjusted[i]

total_diff_adj_ind = total_re_adj - total_ind
print("-" * 110) 
print(f"{'TOTAL':<8} {total_ind:<12.2f} {total_re:<12.2f} {total_re_adj:<12.2f} {total_hol:<12.2f} {total_price_weighted_usage:<12.2f} {np.sum(loss_per_player):<12.2f} {total_diff_adj_ind:<12.2f}")

AGGREGATOR LOSS ANALYSIS
Individual Participation Profit 2162561.1510556294
Expected Replay Profit 2214694.350689539
Total loss across all time periods: 1463.26
Realized Profit 2216157.614109417
Holistic Profit 2215604.2461897717

INDIVIDUAL PROFIT ANALYSIS
Total price-weighted usage: 34716844.02
Player   Individual   Replay       Re+Loss      Holistic     Price Weight Loss Share   Re+Loss-Ind 
--------------------------------------------------------------------------------------------------------------
0        227124.41    232189.58    232306.01    232206.36    2762452.46   116.43       5181.60     
1        337825.42    346140.60    346390.86    346201.84    5937669.17   250.26       8565.44     
2        191347.20    198099.95    198220.85    198159.63    2868374.97   120.90       6873.65     
3        53638.71     55514.86     55552.27     55632.04     887545.32    37.41        1913.56     
4        402108.25    410507.26    410812.56    410519.23    7243245.87   305.29       8704

In [ ]:
print("="*80) ; print("DETAILED AGGREGATOR LOSS DEBUGGING") ; print("="*80)

total_losses = [] ; detailed_results = []

for t in range(T):
    scenario_losses = [] ; scenario_details = []
    
    for s in range(S):
        total_supply = np.sum(dp_re[:, t, s]) ; total_demand = np.sum(dm_re[:, t, s])
        lambda_price = -lambda_rep[t, s] * S ; rt_price = P_RT[t, s]
        
        imbalance = total_demand - total_supply ; price_diff = lambda_price - rt_price 
        loss = imbalance * price_diff
        
        if imbalance > 0: scenario_type = "Excess Demand"
        elif imbalance < 0: scenario_type = "Excess Supply"
        else: scenario_type = "Balanced"
        
        detail = {
            't': t, 's': s,
            'total_demand': total_demand,
            'total_supply': total_supply,
            'imbalance': imbalance,
            'lambda_price': lambda_price,
            'rt_price': rt_price,
            'price_diff': price_diff,
            'loss': loss,
            'scenario_type': scenario_type
        }
        
        scenario_losses.append(loss) ; scenario_details.append(detail)
    
    avg_loss = np.mean(scenario_losses) ; total_losses.append(avg_loss) ; detailed_results.extend(scenario_details)

total_loss = np.sum(total_losses) ; print(f"TOTAL LOSS: {total_loss:.2f}")
print(f"Average loss per period: {np.mean(total_losses):.2f}") ; print()

TARGET_TIMES = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19, 20, 21, 22,23]

if TARGET_TIMES is not None: display_periods = TARGET_TIMES ; print(f"DETAILED BREAKDOWN (Time periods: {TARGET_TIMES}):")
print("-" * 120) ; print(f"{'t':<3} {'s':<3} {'Demand':<8} {'Supply':<8} {'Imbal':<8} {'Lambda':<8} {'RT':<8} {'L-RT':<8} {'Loss':<10} {'Type':<15}") ; print("-" * 120)

for detail in detailed_results:
    if detail['t'] in display_periods:
        print(f"{detail['t']:<3} {detail['s']:<3} {detail['total_demand']:<8.2f} {detail['total_supply']:<8.2f} "
              f"{detail['imbalance']:<8.2f} {detail['lambda_price']:<8.2f} {detail['rt_price']:<8.2f} "
              f"{detail['price_diff']:<8.2f} {detail['loss']:<10.2f} {detail['scenario_type']:<15}")
print()

excess_demand_losses = [d['loss'] for d in detailed_results if d['scenario_type'] == 'Excess Demand']
excess_supply_losses = [d['loss'] for d in detailed_results if d['scenario_type'] == 'Excess Supply']
balanced_losses = [d['loss'] for d in detailed_results if d['scenario_type'] == 'Balanced']

print("LOSS BY SCENARIO TYPE:")
print(f"Excess Demand scenarios: {len(excess_demand_losses)} cases, Total loss: {sum(excess_demand_losses):.2f}")
print(f"Excess Supply scenarios: {len(excess_supply_losses)} cases, Total loss: {sum(excess_supply_losses):.2f}")
print(f"Balanced scenarios: {len(balanced_losses)} cases, Total loss: {sum(balanced_losses):.2f}")
print()

lambda_gt_rt_count = sum(1 for d in detailed_results if d['price_diff'] > 0.001)
lambda_lt_rt_count = sum(1 for d in detailed_results if d['price_diff'] < -0.001)
lambda_eq_rt_count = sum(1 for d in detailed_results if abs(d['price_diff']) < 0.001)

print("PRICE RELATIONSHIP ANALYSIS:")
print(f"Lambda > RT: {lambda_gt_rt_count} cases ({lambda_gt_rt_count/(T*S)*100:.1f}%)")
print(f"Lambda < RT: {lambda_lt_rt_count} cases ({lambda_lt_rt_count/(T*S)*100:.1f}%)")
print(f"Lambda ≈ RT: {lambda_eq_rt_count} cases ({lambda_eq_rt_count/(T*S)*100:.1f}%)")

print("\nLOSS BY TIME PERIOD:")
for t in range(T): 
    period_losses = [d['loss'] for d in detailed_results if d['t'] == t]
    avg_period_loss = np.mean(period_losses)
    print(f"t={t}: Average loss = {avg_period_loss:.2f}")

DETAILED AGGREGATOR LOSS DEBUGGING
TOTAL LOSS: 1463.26
Average loss per period: 60.97

DETAILED BREAKDOWN (Time periods: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]):
------------------------------------------------------------------------------------------------------------------------
t   s   Demand   Supply   Imbal    Lambda   RT       L-RT     Loss       Type           
------------------------------------------------------------------------------------------------------------------------
0   0   0.00     0.00     0.00     213.44   60.74    152.71   0.00       Balanced       
0   1   0.00     0.00     0.00     213.44   74.43    139.02   0.00       Balanced       
0   2   0.00     0.00     0.00     213.44   41.92    171.53   0.00       Balanced       
0   3   0.00     0.00     0.00     213.44   55.56    157.88   0.00       Balanced       
0   4   0.00     0.00     0.00     213.44   48.54    164.91   0.00       Balanced       
0   5   0.00  

In [ ]:
print("="*100) ; print("INDIVIDUAL vs REPLAY PROFIT COMPONENT COMPARISON") ; print("="*100)

components = ['DA_Market', 'RT_Market', 'Penalty', 'Pool_Supply', 'Pool_Demand']
profit_ind_comp = {comp: np.zeros(I) for comp in components}
profit_re_comp = {comp: np.zeros(I) for comp in components}

for i in range(I):
    for t in range(T):
        profit_ind_comp['DA_Market'][i] += P_DA[t] * x_ind[i, t]
    for t in range(T):
        profit_ind_comp['RT_Market'][i] += np.mean([P_RT[t, s] * yp_ind[i, t, s] for s in range(S)])
    for t in range(T):
        profit_ind_comp['Penalty'][i] -= np.mean([P_PN[t, s] * ym_ind[i, t, s] for s in range(S)])
    profit_ind_comp['Pool_Supply'][i] = 0
    profit_ind_comp['Pool_Demand'][i] = 0

for i in range(I):
    for t in range(T):
        profit_re_comp['DA_Market'][i] += P_DA[t] * x_re[i, t]
    for t in range(T):
        profit_re_comp['RT_Market'][i] += np.mean([P_RT[t, s] * yp_re[i, t, s] for s in range(S)])
    for t in range(T):
        profit_re_comp['Penalty'][i] -= np.mean([P_PN[t, s] * ym_re[i, t, s] for s in range(S)])
    for t in range(T):
        lambda_price = -lambda_rep[t, :] * S
        profit_re_comp['Pool_Supply'][i] += np.mean([lambda_price[s] * dp_re[i, t, s] for s in range(S)])
    for t in range(T):
        lambda_price = -lambda_rep[t, :] * S
        profit_re_comp['Pool_Demand'][i] -= np.mean([lambda_price[s] * dm_re[i, t, s] for s in range(S)])

print("INDIVIDUAL PARTICIPATION COMPONENTS:")
print(f"{'Player':<8} {'DA Market':<12} {'RT Market':<12} {'Penalty':<12} {'Pool Supply':<12} {'Pool Demand':<12} {'Total':<12}")
print("-" * 85)

total_ind_comp = {comp: 0 for comp in components}
for i in range(I):
    player_total_ind = sum(profit_ind_comp[comp][i] for comp in components)
    print(f"{i:<8} {profit_ind_comp['DA_Market'][i]:<12.2f} {profit_ind_comp['RT_Market'][i]:<12.2f} "
          f"{profit_ind_comp['Penalty'][i]:<12.2f} {profit_ind_comp['Pool_Supply'][i]:<12.2f} "
          f"{profit_ind_comp['Pool_Demand'][i]:<12.2f} {player_total_ind:<12.2f}")
    
    for comp in components:
        total_ind_comp[comp] += profit_ind_comp[comp][i]

print("-" * 85)
total_ind_all = sum(total_ind_comp[comp] for comp in components)
print(f"{'TOTAL':<8} {total_ind_comp['DA_Market']:<12.2f} {total_ind_comp['RT_Market']:<12.2f} "
      f"{total_ind_comp['Penalty']:<12.2f} {total_ind_comp['Pool_Supply']:<12.2f} "
      f"{total_ind_comp['Pool_Demand']:<12.2f} {total_ind_all:<12.2f}")

print("\n" + "="*85)
print("REPLAY PARTICIPATION COMPONENTS:")
print(f"{'Player':<8} {'DA Market':<12} {'RT Market':<12} {'Penalty':<12} {'Pool Supply':<12} {'Pool Demand':<12} {'Total':<12}")
print("-" * 85)

total_re_comp = {comp: 0 for comp in components}
for i in range(I):
    player_total_re = sum(profit_re_comp[comp][i] for comp in components)
    print(f"{i:<8} {profit_re_comp['DA_Market'][i]:<12.2f} {profit_re_comp['RT_Market'][i]:<12.2f} "
          f"{profit_re_comp['Penalty'][i]:<12.2f} {profit_re_comp['Pool_Supply'][i]:<12.2f} "
          f"{profit_re_comp['Pool_Demand'][i]:<12.2f} {player_total_re:<12.2f}")
    
    for comp in components:
        total_re_comp[comp] += profit_re_comp[comp][i]

print("-" * 85)
total_re_all = sum(total_re_comp[comp] for comp in components)
print(f"{'TOTAL':<8} {total_re_comp['DA_Market']:<12.2f} {total_re_comp['RT_Market']:<12.2f} "
      f"{total_re_comp['Penalty']:<12.2f} {total_re_comp['Pool_Supply']:<12.2f} "
      f"{total_re_comp['Pool_Demand']:<12.2f} {total_re_all:<12.2f}")

# Difference analysis
print("\n" + "="*85)
print("DIFFERENCE (REPLAY - INDIVIDUAL):")
print(f"{'Player':<8} {'DA Market':<12} {'RT Market':<12} {'Penalty':<12} {'Pool Supply':<12} {'Pool Demand':<12} {'Total':<12}")
print("-" * 85)

for i in range(I):
    diff_da = profit_re_comp['DA_Market'][i] - profit_ind_comp['DA_Market'][i]
    diff_rt = profit_re_comp['RT_Market'][i] - profit_ind_comp['RT_Market'][i]
    diff_penalty = profit_re_comp['Penalty'][i] - profit_ind_comp['Penalty'][i]
    diff_pool_supply = profit_re_comp['Pool_Supply'][i] - profit_ind_comp['Pool_Supply'][i]
    diff_pool_demand = profit_re_comp['Pool_Demand'][i] - profit_ind_comp['Pool_Demand'][i]
    diff_total = diff_da + diff_rt + diff_penalty + diff_pool_supply + diff_pool_demand
    
    print(f"{i:<8} {diff_da:<12.2f} {diff_rt:<12.2f} {diff_penalty:<12.2f} "
          f"{diff_pool_supply:<12.2f} {diff_pool_demand:<12.2f} {diff_total:<12.2f}")

print("-" * 85)
total_diff_da = total_re_comp['DA_Market'] - total_ind_comp['DA_Market']
total_diff_rt = total_re_comp['RT_Market'] - total_ind_comp['RT_Market']
total_diff_penalty = total_re_comp['Penalty'] - total_ind_comp['Penalty']
total_diff_pool_supply = total_re_comp['Pool_Supply'] - total_ind_comp['Pool_Supply']
total_diff_pool_demand = total_re_comp['Pool_Demand'] - total_ind_comp['Pool_Demand']
total_diff_all = total_re_all - total_ind_all

print(f"{'TOTAL':<8} {total_diff_da:<12.2f} {total_diff_rt:<12.2f} {total_diff_penalty:<12.2f} "
      f"{total_diff_pool_supply:<12.2f} {total_diff_pool_demand:<12.2f} {total_diff_all:<12.2f}")

# Summary analysis
print("\n" + "="*60)
print("[AGGREGATION BENEFIT ANALYSIS]")
print("="*60)
print(f"Total Individual Profit:     {total_ind_all:.2f}")
print(f"Total Replay Profit:         {total_re_all:.2f}")
print(f"Aggregation Benefit:         {total_diff_all:.2f}")
print(f"Benefit Percentage:          {(total_diff_all/total_ind_all)*100:.2f}%")

print(f"\nComponent-wise Benefits:")
print(f"DA Market Change:            {total_diff_da:.2f}")
print(f"RT Market Change:            {total_diff_rt:.2f}")
print(f"Penalty Reduction:           {total_diff_penalty:.2f}")
print(f"Pool Supply Revenue:         {total_diff_pool_supply:.2f}")
print(f"Pool Demand Cost:            {total_diff_pool_demand:.2f}")
print(f"Net Pool Benefit:            {total_diff_pool_supply + total_diff_pool_demand:.2f}")

# Verification
print(f"\n[VERIFICATION]")
print(f"Expected Individual Total:   {OBJ_IND:.2f}")
print(f"Expected Replay Total:       {OBJ_RE:.2f}")
print(f"Calculated Individual Total: {total_ind_all:.2f}")
print(f"Calculated Replay Total:     {total_re_all:.2f}")

INDIVIDUAL vs REPLAY PROFIT COMPONENT COMPARISON
INDIVIDUAL PARTICIPATION COMPONENTS:
Player   DA Market    RT Market    Penalty      Pool Supply  Pool Demand  Total       
-------------------------------------------------------------------------------------
0        25894.63     204118.09    -2888.30     0.00         0.00         227124.41   
1        45871.64     295713.71    -3759.93     0.00         0.00         337825.42   
2        23503.47     170831.93    -2988.20     0.00         0.00         191347.20   
3        3858.72      50150.76     -370.76      0.00         0.00         53638.71    
4        56302.32     351872.23    -6066.31     0.00         0.00         402108.25   
5        4366.25      92121.07     -631.67      0.00         0.00         95855.65    
6        5598.03      85646.69     -591.31      0.00         0.00         90653.41    
7        70248.80     400602.54    -6248.40     0.00         0.00         464602.94   
8        10460.24     124717.70    -1031.69  

In [ ]:
# Profit Component Summary Table
import pandas as pd
import numpy as np

# 기존 계산된 값들 사용 (total_ind_comp, total_re_comp, total_loss)
# Individual vs Replay 차이 계산
total_diff_da = total_re_comp['DA_Market'] - total_ind_comp['DA_Market']
total_diff_penalty = total_re_comp['Penalty'] - total_ind_comp['Penalty']
total_diff_rt = total_re_comp['RT_Market'] - total_ind_comp['RT_Market']
total_pool = total_re_comp['Pool_Supply'] + total_re_comp['Pool_Demand']

# 요약 테이블 생성
summary_data = {
    'Scenario': ['Individual', 'Replay', 'Difference'],
    'DA': [
        total_ind_comp['DA_Market'],
        total_re_comp['DA_Market'], 
        total_diff_da
    ],
    'Penalty': [
        total_ind_comp['Penalty'],
        total_re_comp['Penalty'],
        total_diff_penalty
    ],
    'Real-time': [
        total_ind_comp['RT_Market'],
        total_re_comp['RT_Market'],
        total_diff_rt
    ],
    'Pool': [
        0,  # Individual에는 pool 없음
        total_pool,
        total_pool
    ],
    'Loss': [
        0,  # Individual에는 loss 없음
        total_loss,
        total_loss
    ],
    'Total': [
        sum(total_ind_comp[comp] for comp in ['DA_Market', 'RT_Market', 'Penalty']),
        sum(total_re_comp[comp] for comp in ['DA_Market', 'RT_Market', 'Penalty']) + total_pool + total_loss,
        total_diff_da + total_diff_penalty + total_diff_rt + total_pool + total_loss
    ]
}

# DataFrame 생성
df_summary = pd.DataFrame(summary_data)

# 소수점 2자리로 반올림
for col in ['DA', 'Penalty', 'Real-time', 'Pool', 'Loss', 'Total']:
    df_summary[col] = df_summary[col].round(2)

print("="*80)
print("PROFIT COMPONENT SUMMARY TABLE")
print("="*80)
print(df_summary.to_string(index=False))

# 요약 테이블 생성 (수정)
da_penalty_ind = total_ind_comp['DA_Market'] + total_ind_comp['Penalty']
da_penalty_re = total_re_comp['DA_Market'] + total_re_comp['Penalty']
da_penalty_diff = da_penalty_re - da_penalty_ind

pool_loss_re = total_pool + total_loss
pool_loss_diff = pool_loss_re

summary_data = {
    'Scenario': ['Individual', 'Replay', 'Difference'],
    'DA-Penalty': [
        da_penalty_ind,
        da_penalty_re,
        da_penalty_diff
    ],
    'Real-time': [
        total_ind_comp['RT_Market'],
        total_re_comp['RT_Market'],
        total_diff_rt
    ],
    'Pool-Loss': [
        0,
        pool_loss_re,
        pool_loss_diff
    ],
    'Total': [
        da_penalty_ind + total_ind_comp['RT_Market'],
        da_penalty_re + total_re_comp['RT_Market'] + pool_loss_re,
        da_penalty_diff + total_diff_rt + pool_loss_diff
    ]
}

# DataFrame 생성
df_summary = pd.DataFrame(summary_data)

# 소수점 2자리로 반올림
for col in ['DA-Penalty', 'Real-time', 'Pool-Loss', 'Total']:
    df_summary[col] = df_summary[col].round(2)

print("="*60)
print("PROFIT COMPONENT SUMMARY TABLE")
print("="*60)
print(df_summary.to_string(index=False))

PROFIT COMPONENT SUMMARY TABLE
  Scenario         DA    Penalty   Real-time       Pool     Loss       Total
Individual 270741.950 -27301.250 1919120.450      0.000    0.000 2162561.150
    Replay 524463.820 -22044.510 1535252.720 177022.330 1463.260 2216157.610
Difference 253721.870   5256.740 -383867.730 177022.330 1463.260   53596.460
PROFIT COMPONENT SUMMARY TABLE
  Scenario  DA-Penalty   Real-time  Pool-Loss       Total
Individual  243440.700 1919120.450      0.000 2162561.150
    Replay  502419.300 1535252.720 178485.590 2216157.610
Difference  258978.600 -383867.730 178485.590   53596.460
